# Play around with coalition weights and summed coalition weights

In [ ]:
import itertools as it
import random
from math import prod

import numpy as np
from scipy.special import comb
from tqdm import tqdm

random.seed(0)

In [ ]:
num_rows = 12
show_digits = 3


def to_str(val):
    if isinstance(val, float):
        return f"{val:.{show_digits}g}"
    else:
        return str(val)


def align_fstr(align):
    match align:
        case "center":
            return "^"
        case "right":
            return ">"
        case "left":
            return "<"
        case _:
            raise ValueError(f"Invalid alignment: {align}")


def format(val, width, align="center"):
    align = align_fstr(align)
    if isinstance(val, float):
        return f"{val:{align}{width}.{show_digits}g}"
    else:
        return f"{val:{align}{width}}"


def print_triangle(rows, with_sum=True, with_len=True, to_str=to_str, format_fn=format):
    max_digits = max(len(to_str(c)) for row in rows for c in row)
    max_width = len(rows[-1]) * max_digits + len(rows[-1]) - 1

    if with_len:
        row_lens = [len(row) for row in rows]
        row_len_width = max(len(to_str(row_len)) for row_len in row_lens)

    if with_sum:
        rowsums = [sum(row) for row in rows]
        rowsum_width = max(len(to_str(rowsum)) for rowsum in rowsums)

    for i, row in enumerate(rows):
        numbers = " ".join(format_fn(c, max_digits) for c in row)
        line = numbers.center(max_width)
        if with_len:
            len_str = format_fn(row_lens[i], row_len_width, align="left")
            line = len_str + " " * 4 + line
        if with_sum:
            rowsum_str = format_fn(rowsums[i], rowsum_width, align="right")
            line += " " * 4 + rowsum_str
        line = f"{i:>{len(str(len(rows) - 1))}}" + " │ " + line
        print(line)


## Pascal's Triangle

In [ ]:
pt_rows = [
    [comb(n, k, exact=True) for k in range(n + 1)] for n in tqdm(range(num_rows))
]

In [ ]:
print_triangle(pt_rows)

## Coalition Weights

$$
\frac{1}{n} {n \choose k}^{-1}
$$

In [ ]:
coali_ws = [
    [1 / ((n + 1) * comb(n, k, exact=True)) for k in range(n + 1)]
    for n in tqdm(range(num_rows))
]

In [ ]:
print_triangle(coali_ws)

## Sum of Coalition Weights, Grouped by Coalition Size

Multiply coalition weights by the number of coalitions of that size.

$$
\frac{1}{n} {n \choose k} {n \choose k}^{-1} = \frac{1}{n}
$$

In [ ]:
grouped_coali_ws = [
    [
        1 / ((n + 1) * comb(n, k, exact=True)) * comb(n, k, exact=True)
        for k in range(n + 1)
    ]
    for n in tqdm(range(num_rows))
]

In [ ]:
print_triangle(grouped_coali_ws)

# Branches in Shapley BaB
We now change our perspective: Instead of the number of permutations for sets of increasing size, we now look at a set of a fixed size that is splitted into branches as in Shapley BaB (as done in src/shap_bounds.py in the version from the 27th of August 2025).

In [ ]:
num_features = 10

Branches are formed by including/excluding individual features.

In [ ]:
feature_order = list(range(num_features))
# random.shuffle(feature_order)

In [ ]:
# Include/Exclude Masks.
# - If the first element is True, then the feature is included.
# - If the second element is False, then the feature is excluded.
# - If the first is False and the second is True, the branch contains sets
#   that contain the feature, as well as sets that do not.
# - First element True, second False is invalid.
root_branch = (np.zeros(num_features, dtype=bool), np.ones(num_features, dtype=bool))
levels = [(root_branch,)]


def include(branch, feature):
    lb, ub = branch[0].copy(), branch[1].copy()
    lb[feature] = ub[feature] = True
    return (lb, ub)


def exclude(branch, feature):
    lb, ub = branch[0].copy(), branch[1].copy()
    lb[feature] = ub[feature] = False
    return (lb, ub)


for i in tqdm(feature_order):
    new_branches = [(exclude(b, i), include(b, i)) for b in levels[-1]]
    new_branches = list(it.chain(*new_branches))
    levels.append(new_branches)


In [ ]:
def branch_str(val):
    if not isinstance(val, tuple):
        return to_str(val)
    lb, ub = val
    return "".join(
        "+" if l else "-" if not u else "?" for l, u in zip(lb, ub, strict=True)
    )


def format_branch(val, width, align="center"):
    if not isinstance(val, tuple):
        return format(val, width, align)
    else:
        branch = val
        align = align_fstr(align)
        str_ = branch_str(branch)
        return f"{str_:{align}{width}}"

In the plot below, each position of a `+`/`-`/`?` string refers to a feature.
A `+` means that all sets in a branch include the feature at this position, a `-` means it is excluded, and `?` means the branch contains both sets that contain and do not contain this feature.

In [ ]:
print_triangle(levels, to_str=branch_str, format_fn=format_branch, with_sum=False)

In [ ]:
def num_determined(branch):
    return branch[0].sum() + (branch[1] == False).sum()  # noqa: E712


determined_counts = [[num_determined(b) for b in bs] for bs in levels]
print_triangle(determined_counts)

## Sums of Coalition Weights
Now it gets interesting: We compute the sum of coalition weights in each branch
$$
\frac{1}{n} \sum_{k=0}^{n-1-t} {n-1-t \choose k} {n-1 \choose k + r}^{-1},
$$
where $r$ is the number of elements excluded in the branch and $t$ is the number of included and excluded elements in the branch.
Here we ignore that there is one feature for which we want to compute the influence, so `n-1 = num_features` in the above formula.

In [ ]:
def num_included(branch):
    return branch[0].sum()


def branch_sum_coali_weights(branch):
    r = num_included(branch)
    t = num_determined(branch)
    res = sum(
        comb(num_features - t, k) * 1 / comb(num_features, k + r)
        for k in range(num_features - t + 1)  # +1 to include the endpoint
    )
    return 1 / (num_features + 1) * res


sum_coali_ws_rows = [[branch_sum_coali_weights(b) for b in bs] for bs in tqdm(levels)]


In [ ]:
print_triangle(sum_coali_ws_rows)

There is a pattern in this data and it relates to the number of included elements in a branch.

In [ ]:
rows_num_included = [[num_included(b) for b in bs] for bs in tqdm(levels)]


In [ ]:
print_triangle(rows_num_included)

Or, to be precise:

In [ ]:
rows_num_included_mod = [
    [r if r <= i // 2 else i - r for r in rs] for i, rs in enumerate(rows_num_included)
]


In [ ]:
print_triangle(sum_coali_ws_rows)

In [ ]:
print_triangle(rows_num_included_mod, with_sum=False)

Check this rule for all entries of levels.

In [ ]:
for level in tqdm(levels):
    cache = {}
    for i, branch in enumerate(level):
        r = num_included(branch)
        r_ = r if r <= i // 2 else i - r
        c = branch_sum_coali_weights(branch)

        if r in cache:
            assert cache[r] == c, f"r={r} in cache, but {cache[r]=} != {c=}"
        else:
            cache[r] = c

Now reproduce these numbers with a more simple formula.

In [ ]:
def num_excluded(branch):
    return (branch[1] == False).sum()  # noqa: E712


def branch_sum_coali_weights_repro1(branch):
    r = num_included(branch)
    s = num_excluded(branch)
    t = num_determined(branch)
    assert r + s == t

    res = sum(
        prod(k + i for i in range(1, r + 1))
        * prod(num_features - r - k - i for i in range(s))
        for k in range(num_features - t + 1)  # +1 to include the endpoint
    )
    return res / prod(range(num_features + 1 - t, num_features + 2))


sum_coali_ws_repro1_rows = [
    [branch_sum_coali_weights_repro1(b) for b in bs] for bs in tqdm(levels)
]

In [ ]:
print("True")
print()

print_triangle(sum_coali_ws_rows)

In [ ]:
print("Reproduced")
print()

print_triangle(sum_coali_ws_repro1_rows)

Much better than a visual check:

In [ ]:
for true_row, repro_row in zip(
    sum_coali_ws_rows, sum_coali_ws_repro1_rows, strict=True
):
    for true, repro in zip(true_row, repro_row, strict=True):
        assert np.isclose(true, repro), f"{true=}, {repro=}"


## Reduced Formula from ChatGPT
ChatGPT figured out a formula based on the Hamming weight of the index of each branch in the respective layer.
This Hamming weight is just the number of included entries.
With this, the formula from ChatGPT is
$$
\frac{1}{n} \sum_{k=0}^{n-1-t} {n-1-t \choose k} {n-1 \choose k + r}^{-1} = \frac{1}{(t+1){t \choose r}},
$$
where $t$ and $r$ are as above.
Let's check this formula.

In [ ]:
def by_hamming_weights(branch):
    t = num_determined(branch)
    r = num_included(branch)
    return 1 / ((t + 1) * comb(t, r))


by_hamming_weights = [[by_hamming_weights(b) for b in bs] for bs in tqdm(levels)]


In [ ]:
print_triangle(by_hamming_weights)

In [ ]:
for true_row, repro_row in zip(sum_coali_ws_rows, by_hamming_weights, strict=True):
    for true, repro in zip(true_row, repro_row, strict=True):
        assert np.isclose(true, repro), f"{true=}, {repro=}"

Wow!

Also check the recursive formula, just to be sure.

In [ ]:
# (coalitions weight sub, depth, number of included)
vals_levels = [((1, 0, 0),)]


def refine_exclude(a, t, r):
    a_ = (t + 1 - r) / (t + 2) * a
    return (a_, t + 1, r)


def refine_include(a, t, r):
    a_ = (r + 1) / (t + 2) * a
    return (a_, t + 1, r + 1)


for _ in tqdm(feature_order):
    new_vals = [
        (refine_exclude(*v), refine_include(*v))
        for v in vals_levels[-1]
    ]
    new_vals = list(it.chain(*new_vals))
    vals_levels.append(new_vals)

sum_coali_ws_repro2_rows = [[a for a, _, _ in vals] for vals in vals_levels]

In [ ]:
print("True")
print()

print_triangle(sum_coali_ws_rows)

In [ ]:
print("Reproduced")
print()

print_triangle(sum_coali_ws_repro2_rows)

In [ ]:
for true_row, repro_row in zip(sum_coali_ws_rows, sum_coali_ws_repro2_rows, strict=True):
    for true, repro in zip(true_row, repro_row, strict=True):
        assert np.isclose(true, repro), f"{true=}, {repro=}"